# newnucal on real HERA data — `zen.LST.mini.131_nights.FR0filt.uvh5`

Applies **newnucal** forward-model spectral-redundancy calibration to a small subset
(a few integrations × one sub-band) of the 131-night LST stack: validate the pipeline on
real data, measure fitting speed, and push residual delay spectra down to expose low-level
chromatic structure (mutual coupling, cable reflections, high-RM sources).

**v2 changes** (after the first executed run):
- The **FR0/crosstalk notch is handled by an explicit subspace choice**
  (`PROJECT_FR0`): with the notch at `xtalk_frate = 0.025 mHz`, sky signal is only removed
  where fringe rates are that slow — `|b_EW| ≲ 1 m` at 125 MHz, i.e. pure N/S baselines
  (plus near-polar sky everywhere). Option A (default): `project_out_time_mean=True`
  removes the per-(baseline, channel) time-mean from data and model — a wider notch
  (~1/T ≈ 0.19 mHz) that also guards against residual sub-notch systematics, at the cost
  of 1/ntime of the data DOF and most sky signal on `|b_EW| ≲ 5 m` baselines. Option B:
  no projector, just zero-weight `|b_EW| < 3 m` baselines where the real notch bit.
- **Dead channels** — channels fully flagged at the selected times (FM band, band edges,
  the 125 MHz = 250/2 clock line, transient RFI) — are excluded from the objective via
  channel weights, and gains are *interpolated* across them after each solve; the v1 run
  left raw-model spikes there that poisoned the delay spectra.
- **Stage B is sky-only** (beam frozen): with beam and gains fixed the model is exactly
  linear in the sky coefficients, so the solver choice is `SKY_SOLVER='cg'`
  (preconditioned conjugate gradient on the normal equations — `Calibrator.fit_sky_cg`)
  or `'dirty'` (the adjoint/Richardson sweeps). Stage C is the original **joint
  sky+beam L-BFGS** (`fit_joint_sky_beam_lbfgs`) — watch the external χ²: the v1 run
  showed the joint fit can scramble the weakly-constrained beam while internal loss falls,
  so the result is accepted only if external χ² improves.
- **Positivity** is enforced between stages by a POCS projection
  (`realdata.project_positive`), fixing the negative-pixel skies.
- **Noise σ** is predicted from the uvh5 autocorrelations and `nsamples` using
  `hera_cal.noise.predict_noise_variance_from_autos`, rather than an empirical
  model-error floor.
- Speed experiments moved to `newnucal/benchmarks/benchmark_real_data_blocks.py`.
- **v7: `NSIDE_SKY=64`** — the resolution taper now admits baselines to ~75 m; the GSM
  cube (stored at nside 32) is flux-conserving upsampled (`ud_grade`, `power=-2`) to any
  `NSIDE_SKY` instead of asserting equality.
- **v6: wide-band channel selection.** Spectral-redundancy calibration needs bandwidth
  lever arm: `CHANNELS` now spreads a fixed channel count across the active band
  (108–225 MHz) instead of a contiguous sliver. The 2D path is per-channel exact, so
  decimation is safe for fitting; delay diagnostics alias beyond ±1/(2Δν).
  External χ² now honors the visibility weights (`use_visibility_weights=True`) — under
  the resolution taper the unweighted statistic was dominated by baselines the fit was
  told to ignore, mis-rejecting good L-BFGS results.
- **v5: the model's resolution limit is enforced.** An nside-32 pixel grid only models
  baselines out to `ell ~ RES_ELL_FACTOR x NSIDE` (b ≲ 40 m here); beyond that,
  pixelization noise makes |model| ≫ |data|, and any |model|²-weighted amplitude solve
  (the bootstrap, the per-channel gains) gets dragged orders of magnitude down — the
  "model far below data, wide in delay" failure. The package's
  `baseline_resolution_taper` is now folded into the visibility weights; longer baselines
  re-enter when `NSIDE_SKY` increases (and point sources move to exact coordinates).
- **v4: the sky is anchored and the gauge is consistent.** An *amplitude bootstrap*
  folds the data-preferred smooth scale into the flux cube up front, so the GSM prior,
  the zero gain gauge, and the frozen static sky all agree (without this, the gauge
  re-pin and the prior fight each other and the loss explodes). The v3 run produced speckle at 30× the GSM peak,
  concentrated in low-weight pixels: with no prior, χ² is flat along weakly-constrained
  directions and preconditioned solvers (CG's beam diagonal, L-BFGS's diag-GN) walk them
  as fast as strong ones — solution variance ∝ 1/weight — while the per-(t,ν) gain
  freedom leaves the overall sky scale a pure degeneracy that ratchets between solves.
  Now: (i) pixels below `SKY_WEIGHT_CUT` of the peak apparent-beam weight are *not fitted*
  — their GSM flux is simulated once and subtracted from the data (protects the L-BFGS
  stage too); (ii) `fit_sky_cg` solves ridge-regression toward the GSM init
  (`PRIOR_LAMBDA`), pinning both the weakly-constrained directions and the amplitude scale.

## Data caveats (from the file header/history)
LST-stacked 131 nights, **redundantly averaged** (one pair per group), **FR0-notched**
(`FR0_HALFWIDTH=0.01 mHz`, autos exempt), flagged channels **re-inpainted** (`nsamples==0`
marks them; values are pipeline-smooth and usable for plots but not for fitting). Only
auto present is `(0,0)`. Data must be **conjugated** to match newnucal's fringe sign.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import jax
jax.config.update('jax_enable_x64', True)   # newnucal.utils uses float64
import jax.numpy as jnp
import jaxopt
import healpy
from astropy.time import Time
from hera_cal.datacontainer import DataContainer
from hera_cal.noise import predict_noise_variance_from_autos

from newnucal import BeamModel, BeamBasis, SkyModel, SkyBasis, Calibrator
from newnucal.simulate import compute_rotation_matrices
from newnucal.realdata import (
    load_lst_stack_subset, snap_to_hex_lattice, hera_array_from_file_pairs,
    dead_channel_mask, dead_channel_log_weights,
    interp_gains_over_dead_channels, project_positive,
)

# ------------------------------- configuration -------------------------------
DATA_FILE = 'zen.LST.mini.131_nights.FR0filt.uvh5'
SKY_CUBE  = 'newnucal/notebooks/sky_cube_cache_nside32_nfreq64_n5000.npz'

NTIMES_FIT = 8
# Spectral redundancy lives in BANDWIDTH: a fixed number of channels spread
# across the active band (above FM at 108 MHz, below the GSM-cube limit at
# 225 MHz) beats a contiguous sliver. With METHOD='2d' every channel is an
# independent exact NUFFT, so uniform decimation is safe for the model; the
# trade is delay-domain diagnostics alias beyond +-1/(2*channel spacing) —
# for high-delay work (cable reflections) switch to a contiguous range.
CH_STEP = 8                # x 122.07 kHz
CHANNELS = np.arange(504, 1459, CH_STEP)    # 108.5-224.7 MHz
NCHAN = len(CHANNELS)
POL_IDX = 0
T_CHUNK_SIZE = 1   # keep forward-model geometry small; set >= NTIMES_FIT to disable chunking

NSIDE_SKY, NSIDE_BEAM = 64, 16             # nside 64: resolution taper admits b ~ 75 m;
                                            # GSM cube (nside 32) is flux-conserving
                                            # upsampled below
SKY_ETA_MAX  = 50e-9                        # s; INTRINSIC sky spectral structure only —
                                            # power-law spectra over an octave need ~10
                                            # modes; fringe chromaticity comes from the
                                            # NUFFT, not the basis
BEAM_N_MODES = 12                           # wide band needs more beam spectral freedom
BEAM_DIAMETERS = np.linspace(12.0, 18.0, 13)

SKY_SOLVER = 'cg'         # 'cg' (preconditioned conjugate gradient on the exactly-linear
                          # beam-frozen sky problem) or 'dirty' (Richardson/adjoint sweeps)
CG_MAXITER = 20           # CG iterations per outer (1 CG iter = 1 forward + 1 adjoint)
PROJECT_FR0 = True        # True: P(x)=x-mean_t(x) on data+model (wide-notch insurance);
                          # False: no projector, zero-weight |b_EW| < EW_MIN_M instead
EW_MIN_M = 3.0
SKY_WEIGHT_CUT = 1e-4     # fit pixels with apparent beam weight > this x max. The weight
                          # is beam POWER SQUARED, so 1e-4 ~ beam -20 dB (~first sidelobe);
                          # 0.01 kept only the inner ~8 deg (164 pixels) and starved the fit.
                          # the rest keep their GSM flux (frozen, subtracted from data).
                          # chi2 is FLAT along low-weight pixel directions, and the
                          # preconditioned solvers walk them as fast as strong ones, so
                          # unpinned low-weight pixels converge to amplified noise.
RES_ELL_FACTOR = 3.0      # baseline-resolution taper: model is valid to ell ~ factor*NSIDE
                          # (u ~ ell/2pi). At NSIDE 32 this keeps only b < ~35-50 m — the
                          # honest truth of a 1.8-deg pixel grid. Without it, every
                          # amplitude solve (bootstrap, gains, chi2) is dragged by long
                          # baselines where pixelization noise makes |model| >> |data|.
PRIOR_LAMBDA = 0.05       # ridge strength toward the GSM init in fit_sky_cg (relative to
                          # the normal operator scale); also pins the sky/gain amplitude
                          # degeneracy that let the v3 run ratchet the sky to 30x GSM

N_SKY_OUTER, N_SKY_INNER = 3, 4   # stage B outers (gains re-solved per outer) x dirty inners
                                  # stage B is a warm start: most chi2 reduction comes from
                                  # the joint L-BFGS (which also moves beam + gains), so keep
                                  # this short and spend wall-clock in stage C
SKY_STEP = 0.5
LBFGS_MAXITER = 60 # 120?
METHOD = '2d'
GAIN_KEYS = ('log_amp', 'phase', 'phi')

## 1. Load data, geometry, rotation

Loading via `hera_cal.io.HERAData` partial I/O and the `newnucal.realdata` helpers:
antenna positions are snapped to the exact 14.6/3 m hex lattice (all antennas sit on it to
~5 mm; `redcal.reds_to_antpos` must NOT be used for this), and the `HERAArray` baseline
ordering is overridden to match the file's redundant-group pairs. Fit times are spread
across the 1.5 h LST window for Earth-rotation leverage.

In [ ]:
sub = load_lst_stack_subset(
    DATA_FILE,
    np.unique(np.round(np.linspace(0, 139, NTIMES_FIT)).astype(int)),
    CHANNELS, pol_index=POL_IDX)
data, flags, nsamples = sub['data'], sub['flags'], sub['nsamples']
pairs, freqs, jd = sub['pairs'], sub['freqs'], sub['jd']
dnu = float(freqs[1] - freqs[0])
print(f"{len(jd)} times; {NCHAN} chans {freqs[0]/1e6:.1f}-{freqs[-1]/1e6:.1f} MHz; "
      f"pol {sub['pol']}; {len(pairs)} cross bls")
print('flag frac %.3f | inpainted frac %.4f | exact-zero frac %.4f'
      % (flags.mean(), (nsamples == 0).mean(), (data == 0).mean()))

snapped, snap_info = snap_to_hex_lattice(sub['antpos'])
print('lattice snap: max %.1f mm, rms %.1f mm'
      % (snap_info['max_err_m']*1e3, snap_info['rms_err_m']*1e3))
array = hera_array_from_file_pairs(snapped, pairs)
bl_len = np.linalg.norm(array.bls[:, :2], axis=1)
print(f'{array.nbls} baselines, {bl_len.min():.1f}-{bl_len.max():.1f} m')
print(f'channel spacing {dnu/1e6:.2f} MHz over {(freqs[-1]-freqs[0])/1e6:.0f} MHz; '
      f'delay-domain Nyquist ±{1e9/(2*dnu):.0f} ns (diagnostics alias beyond this)')

rot_matrices = compute_rotation_matrices(Time(jd, format='jd'), sub['location'])

## 2. Weights and noise model

- `vis_weights` zero out flagged / inpainted / missing samples.
- **Dead channels** (zero weight at every time and baseline for the selected subset; the
  permanent ones are the FM band and band edges, plus the 125 MHz = 250/2 clock line and
  transient RFI at some times) are excluded via log-channel weights, so the per-channel
  gain solve never sees them.
- σ is predicted from autocorrelations with `hera_cal.noise.predict_noise_variance_from_autos`,
  using the uvh5 `nsamples` arrays. `Calibrator` currently accepts `(time, freq)` or `(freq,)`
  noise weights, so the per-baseline predictions are reduced with a median over baselines.

In [ ]:
vis_weights = ((~flags) & (nsamples > 0) & (data != 0)).astype(np.float64)
dead = dead_channel_mask(vis_weights)
dead_chans = np.flatnonzero(dead)
log_ch_w = dead_channel_log_weights(vis_weights)
print(f'retained vis fraction {vis_weights.mean():.3f}; dead channels: '
      f'{dead_chans.size} at {np.round(freqs[dead]/1e6, 2)} MHz')

def sigma_from_autos_and_nsamples(sub, pairs, nsamples, flags):
    import warnings

    pol = sub['pol']
    auto_data = DataContainer({(i, j, pol): v for (i, j), v in sub['autos'].items()})
    nsample_data = DataContainer({(int(i), int(j), pol): nsamples[..., bi]
                                  for bi, (i, j) in enumerate(pairs)})
    dt_native = float(np.nanmedian(np.asarray(sub['hd'].integration_time)))
    df_native = float(np.nanmedian(np.ediff1d(np.asarray(sub['hd'].freqs))))

    available_auto_ants = sorted(i for (i, j) in sub['autos'] if i == j)
    if not available_auto_ants:
        raise ValueError("No autocorrelations found in sub['autos']; cannot predict noise from autos.")
    fallback_auto_ant = int(available_auto_ants[0])

    sigma_bl = np.full(nsamples.shape, np.nan, dtype=np.float64)
    n_auto_fallback = 0
    for bi, (i, j) in enumerate(pairs):
        bl = (int(i), int(j), pol)
        needs_fallback = ((bl[0], bl[0]) not in sub['autos']) or ((bl[1], bl[1]) not in sub['autos'])
        n_auto_fallback += int(needs_fallback)
        with np.errstate(divide='ignore', invalid='ignore'):
            var = predict_noise_variance_from_autos(
                bl, auto_data, dt=dt_native, df=df_native, nsamples=nsample_data,
                auto_ant=fallback_auto_ant if needs_fallback else None)
        bad = (~np.isfinite(var)) | (var <= 0) | flags[..., bi] | (nsamples[..., bi] <= 0)
        sigma_bl[..., bi] = np.where(bad, np.nan, np.sqrt(var))
    if n_auto_fallback:
        print(f'used auto_ant={fallback_auto_ant} fallback for {n_auto_fallback}/{len(pairs)} baselines')

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        sigma_tf = np.nanmedian(sigma_bl, axis=2)
        sigma_f = np.nanmedian(sigma_tf, axis=0)
        fallback = np.nanmedian(sigma_f)
    sigma_f = np.where(np.isfinite(sigma_f), sigma_f, fallback)
    sigma_tf = np.where(np.isfinite(sigma_tf), sigma_tf, sigma_f[None, :])
    return sigma_tf, sigma_bl, dt_native, df_native

sigma_eff, sigma_auto_bl, dt_native, df_native = sigma_from_autos_and_nsamples(
    sub, pairs, nsamples, flags)
sigma_auto_f = np.nanmedian(sigma_eff, axis=0)
print(f'auto/nsamples σ: dt={dt_native:.2f} s, native Δν={df_native/1e3:.2f} kHz')

plt.figure(figsize=(9, 3))
plt.semilogy(freqs/1e6, sigma_auto_f, label='auto/nsamples σ (median over time, bl)')
ref_auto = next(iter(sub['autos'].values()))
plt.semilogy(freqs/1e6, np.median(ref_auto, axis=0)/np.sqrt(df_native * dt_native * np.nanmedian(nsamples)),
             alpha=0.5, label='single-auto radiometer scale')
plt.xlabel('MHz'); plt.ylabel('σ [Jy]'); plt.legend(); plt.tight_layout()

## 3. Models and calibrator

Bases built natively on the selected channels (resampling the wide-band npz would break
orthonormality). `PROJECT_FR0` selects the FR0 handling (see header). The projector is *wider* than the
pipeline's 0.025 mHz notch (1/T over this window ≈ 0.19 mHz), so it costs one time-DOF per
(baseline, channel) and most sky signal on `|b_EW| ≲ 5 m` baselines, but it also removes
any sub-notch systematics that survived the pipeline (residual crosstalk).

In [ ]:
beam_basis = BeamBasis.from_beam_diameters(freqs, BEAM_DIAMETERS, nside=NSIDE_BEAM, n_modes=BEAM_N_MODES)
sky_basis  = SkyBasis.from_dpss(freqs, SKY_ETA_MAX)
print(f'beam basis: {beam_basis.nmodes} modes | sky basis: {sky_basis.nmodes} modes over {NCHAN} channels')
beam_model = BeamModel(NSIDE_BEAM, freqs, beam_basis)   # AiryBeam(14.6 m), FIXED unless FIT_BEAM
sky_model  = SkyModel(NSIDE_SKY, freqs, sky_basis)

def build_cal(data_use, sky_keep=None):
    cal = Calibrator(array, beam_model, sky_model, freqs, rot_matrices, data_use,
                     method=METHOD, noise_sigma=sigma_eff, channel_weights=log_ch_w,
                     t_chunk_size=T_CHUNK_SIZE, project_out_time_mean=PROJECT_FR0)
    beam_mask = cal.build_beam_mask_altitude(0.0)
    sky_mask = cal.build_sky_mask_from_beam_pixels(beam_mask)
    if sky_keep is not None:
        sky_mask = sky_mask & sky_keep
    cal.apply_sky_mask(sky_mask)
    cal.apply_beam_mask(beam_mask)
    w = vis_weights
    if not PROJECT_FR0:   # drop only the baselines the 0.025 mHz notch actually touched
        w = w * (np.abs(array.bls[:, 0]) >= EW_MIN_M)[None, None, :]
    # restrict every amplitude-sensitive solve to baselines the pixelized sky
    # can actually model (the package's own resolution taper, kept as a
    # multiplier so flags/zeros are preserved)
    taper = np.asarray(cal.baseline_resolution_taper(ell_factor=RES_ELL_FACTOR))
    cal.set_visibility_weights(w * taper[None, :, :])
    print(f'  forward model time chunk size: {cal.t_chunk_size}')
    print('  taper: %.0f%% of (freq, bl) cells fully retained; effective bmax ~ %.0f m'
          % (100.0 * float((taper > 0.99).mean()),
             float(np.max(bl_len[np.any(taper > 0.5, axis=0)]))))
    return cal

cal = build_cal(jnp.array(data))

### Sky initialization (GSM+GLEAM) and conjugation check

The data need conjugation to match newnucal's fringe sign (`+2πi ν b·s/c` with
`bls = pos_i − pos_j`); confirmed on short baselines, where nside-32 pixels still preserve
fringe coherence (≲ 40 λ).

In [ ]:
cube = np.load(SKY_CUBE)
cube_nside = int(cube['sky_nside'])
lf_c, lf = np.log(cube['freqs_hz']), np.log(freqs)
flux0 = np.exp(np.array([np.interp(lf, lf_c, np.log(np.maximum(px, 1e-12)))
                         for px in cube['flux_true']]))
if cube_nside != NSIDE_SKY:
    # Jy/pixel map: power=-2 conserves the flux sum (each child pixel gets 1/4)
    flux0 = np.stack([healpy.ud_grade(flux0[:, i], nside_out=NSIDE_SKY, power=-2)
                      for i in range(flux0.shape[1])], axis=1)
    print(f'GSM cube upsampled nside {cube_nside} -> {NSIDE_SKY} (flux-conserving); '
          f'note: no structure is added below the {cube_nside} pixel scale — the prior is '
          f'smooth there and the data fill in what they can constrain')
prms = cal.init_params()
prms['sky_coeffs'] = cal.init_sky_from_flux(flux0)

model0 = np.asarray(cal.simulate({k: prms[k] for k in ('sky_coeffs',) + GAIN_KEYS}))
dataP = np.asarray(cal.data)                       # projected data (matches model space)
short = bl_len < 60.0
w = vis_weights * (np.abs(model0) > 0) * short[None, None, :]
def xcorr(d):
    num = np.abs(np.sum(w * d * np.conj(model0)))
    return num / np.sqrt(np.sum(w*np.abs(d)**2) * np.sum(w*np.abs(model0)**2))
c_as_is, c_conj = xcorr(dataP), xcorr(np.conj(dataP))
print(f'corr(P(data), model) = {c_as_is:.3f}   corr(conj, model) = {c_conj:.3f}')
# newnucal uses the HERA/matvis convention (bls = x_j - x_i; see newnucal.array
# and tests/test_convention.py), so the data are used as read.  This is a check,
# not a choice: a better match to the conjugate means a convention bug.
if c_conj > c_as_is:
    raise ValueError(
        f'data correlate better with the conjugate model ({c_conj:.3f} vs {c_as_is:.3f}): '
        'baseline-convention mismatch between the file and newnucal')
print('initial reduced chi2 (auto σ, dead chans excluded): %.2f'
      % cal.calc_reduced_chi2_excluding_chans(
          {k: prms[k] for k in ('sky_coeffs',) + GAIN_KEYS}, exclude_chans=dead_chans, use_visibility_weights=True))

### Two-tier sky: fit only well-constrained pixels

Pixels with apparent-beam weight below `SKY_WEIGHT_CUT` of the peak are removed from the
fit, but their GSM flux is not dropped: it is simulated once and subtracted from the data,
so the fitted pixels only need to explain the remainder. This caps the 1/weight noise
amplification in *every* solver stage, including the joint L-BFGS which has no prior hook.

In [ ]:
def simulate_model_time_chunked(cal, sky_coeffs, t_chunk_size=None):
    import gc

    t_chunk_size = int(t_chunk_size or cal.t_chunk_size)
    sky_active = cal._ensure_sky_is_active(sky_coeffs)
    rot = cal.rot_matrices
    sim_raw = cal.fwd.simulate_2d if cal.method == '2d' else cal.fwd.simulate_3d

    raw_mean = None
    if cal.project_out_time_mean:
        raw_sum = None
        for t0 in range(0, cal.ntime, t_chunk_size):
            t1 = min(t0 + t_chunk_size, cal.ntime)
            raw = np.asarray(sim_raw(sky_active, rot[t0:t1], t_chunk_size=t_chunk_size))
            raw_sum = raw.sum(axis=0) if raw_sum is None else raw_sum + raw.sum(axis=0)
            del raw
            gc.collect()
        raw_mean = raw_sum / float(cal.ntime)

    out = np.empty(tuple(cal.data.shape), dtype=np.complex128)
    for t0 in range(0, cal.ntime, t_chunk_size):
        t1 = min(t0 + t_chunk_size, cal.ntime)
        raw = np.asarray(sim_raw(sky_active, rot[t0:t1], t_chunk_size=t_chunk_size))
        if raw_mean is not None:
            raw = raw - raw_mean[None, :, :]
        out[t0:t1] = raw
        del raw
        gc.collect()
    return out


def fit_gains_linear_time_chunked(cal, sky_coeffs, t_chunk_size=None):
    t_chunk_size = int(t_chunk_size or cal.t_chunk_size)
    data_np = np.asarray(cal.data)
    weights_np = np.asarray(cal._effective_weights())
    X = np.column_stack([
        np.ones(cal.nbls, dtype=np.float64),
        np.asarray(cal.bls[:, 0], dtype=np.float64),
        np.asarray(cal.bls[:, 1], dtype=np.float64),
    ])

    log_amp = np.empty((cal.ntime, cal.nfreq), dtype=np.float64)
    phase = np.empty((cal.ntime, cal.nfreq), dtype=np.float64)
    phi = np.empty((cal.ntime, 2, cal.nfreq), dtype=np.float64)
    eye = 1e-10 * np.eye(3, dtype=np.float64)

    model = simulate_model_time_chunked(cal, sky_coeffs, t_chunk_size=t_chunk_size)
    for t0 in range(0, cal.ntime, t_chunk_size):
        t1 = min(t0 + t_chunk_size, cal.ntime)
        vis = model[t0:t1]
        data_chunk = data_np[t0:t1]
        weights = weights_np[t0:t1]

        amp2 = np.abs(vis) ** 2
        safe = amp2 > 1e-30
        g_opt = np.where(safe, data_chunk * np.conj(vis) / np.maximum(amp2, 1e-30), 1.0 + 0.0j)
        g_opt = np.where(np.abs(g_opt) > 1e-30, g_opt, 1.0 + 0.0j)
        log_g = np.log(g_opt)
        den = weights * amp2

        w_sum = den.sum(axis=2) + 1e-30
        log_amp[t0:t1] = (den * np.real(log_g)).sum(axis=2) / w_sum

        Xy = np.einsum('bk,tfb->tfk', X, den * np.imag(log_g), optimize=True)
        XTX = np.einsum('bk,tfb,bl->tfkl', X, den, X, optimize=True) + eye
        theta = np.linalg.solve(XTX, Xy[..., None])[..., 0]
        phase[t0:t1] = theta[:, :, 0]
        phi[t0:t1] = theta[:, :, 1:].transpose(0, 2, 1)

    return {
        'log_amp': jnp.array(log_amp),
        'phase': jnp.array(phase),
        'phi': jnp.array(phi),
    }, np.nan


# amplitude bootstrap: the GSM x Airy model and the (FR0-projected) data can
# disagree on overall scale by a large smooth-in-frequency factor; fold the
# data-preferred smooth scale into the flux cube BEFORE it becomes the init,
# the prior, and the frozen static sky — otherwise the zero gain gauge, the
# prior, and the static subtraction are mutually inconsistent. This bootstrap
# is streamed over time chunks because the full-sky gain solve is the peak
# memory point for NTIMES_FIT=8, CH_STEP=8.
gp0, _ = fit_gains_linear_time_chunked(cal, cal.init_sky_from_flux(flux0), T_CHUNK_SIZE)
G3 = np.asarray(cal.A_sky)[:, :3]
la_smooth = G3 @ (G3.T @ np.asarray(gp0['log_amp']).mean(axis=0))
flux0 = flux0 * np.exp(la_smooth)[None, :]
print('amplitude bootstrap: smooth scale %.3f - %.3f across band'
      % (float(np.exp(la_smooth).min()), float(np.exp(la_smooth).max())))

w_px = np.asarray(cal.get_sky_beam_weighting())          # full-sky apparent weight
keep = w_px > SKY_WEIGHT_CUT * w_px.max()
sky_init_full = jnp.array(cal.init_sky_from_flux(flux0))
sky_static = jnp.where(jnp.array(keep)[:, None], 0.0, sky_init_full)
vis_static = simulate_model_time_chunked(cal, sky_static, T_CHUNK_SIZE)
print(f'fitting {int(keep.sum())} pixels above weight cut; freezing GSM flux of the rest '
      f'({int((~keep & (w_px > 0)).sum())} low-weight visible pixels)')
cal = build_cal(jnp.array(data) - np.asarray(vis_static), sky_keep=keep)
prms = cal.init_params()
prms['sky_coeffs'] = sky_init_full   # active subset = GSM on kept pixels
healpy.mollview(np.log10(np.maximum(w_px, 1e-6 * w_px.max())), title='apparent beam weight (log10)', sub=121)
healpy.mollview(keep.astype(float), title=f'fitted pixels (cut {SKY_WEIGHT_CUT:g})', sub=122)

### FR0 diagnostic (informational)

Shows how much of each baseline's *model* power lives in the window time-mean — i.e. what
the projector removes per baseline (concentrated at small |EW|, color-coded). Compare with
the pipeline notch, which only touched `|b_EW| ≲ 1 m`.

In [ ]:
# model0 is already projected, so use the raw (unprojected) forward model here
sim_raw = cal.fwd.simulate_2d if METHOD == '2d' else cal.fwd.simulate_3d
raw = np.asarray(sim_raw(cal._ensure_sky_is_active(prms['sky_coeffs']), cal.rot_matrices,
                        t_chunk_size=cal.t_chunk_size))
frac_fr0 = (np.abs(raw.mean(axis=0))**2).mean(axis=0) / np.maximum((np.abs(raw)**2).mean(axis=(0, 1)), 1e-30)
plt.figure(figsize=(8, 3.5))
plt.scatter(array.bls[:, 0], array.bls[:, 1], s=6, c=np.abs(frac_fr0), cmap='coolwarm')
plt.colorbar(label='|EW| / |b|'); plt.xlabel('baseline length [m]')
plt.ylabel('model power fraction removed by FR0 projector'); plt.tight_layout()

## 4. Stage A — closed-form gain solve (sky and beam fixed)

After the solve, gains at dead channels (unconstrained, left at zero) are interpolated in
frequency so the calibrated model stays smooth — v1's "model spikes" at the spur comb came
from exactly this gap.

In [ ]:
t0 = time.time()
gp, loss_A = cal.fit_gains_linear(prms['sky_coeffs'])
gains = {k: jnp.array(v) for k, v in
         interp_gains_over_dead_channels({k: gp[k] for k in GAIN_KEYS}, dead).items()}
ext_A = cal.calc_reduced_chi2_excluding_chans(
    {'sky_coeffs': prms['sky_coeffs'], **gains}, exclude_chans=dead_chans, use_visibility_weights=True)
print('gain solve: %.1fs   ext chi2 %.2f' % (time.time() - t0, ext_A))

model_A = np.asarray(cal.simulate({'sky_coeffs': prms['sky_coeffs'], **gains}))
dataP = np.asarray(cal.data)
fig, axes = plt.subplots(3, 2, figsize=(11, 7), sharex=True)
for row, b in zip(axes, [np.argmin(np.abs(bl_len - L)) for L in (30, 80, 150)]):
    row[0].plot(freqs/1e6, np.abs(dataP[0, :, b]), lw=0.7, label='P(data)')
    row[0].plot(freqs/1e6, np.abs(model_A[0, :, b]), lw=0.7, label='model')
    row[0].set_ylabel(f'|V| bl {tuple(pairs[b])} ({bl_len[b]:.0f} m)')
    row[1].plot(freqs/1e6, np.angle(dataP[0, :, b] * np.conj(model_A[0, :, b])), '.', ms=2)
    row[1].set_ylabel('resid phase')
axes[0, 0].legend(); fig.suptitle('after gains-only solve (t=0, projected quantities)')
fig.tight_layout()

## 5. Stage B — sky-only solve (linear subproblem)

Beam frozen, gains fixed per outer → the model is **exactly linear** in the sky
coefficients. `SKY_SOLVER='cg'` runs preconditioned CG on the normal equations (the
preconditioner is the same per-(pixel, ν) apparent-beam-power diagonal the dirty update
uses); `'dirty'` runs the adjoint sweeps. Each outer ends with a POCS positivity
projection and a gain re-solve. The gain solve is a *log-space surrogate* (it can raise
the true loss when residual phases wrap, as seen in v1), so a gain update is kept only if
it lowers the external χ².

In [ ]:
sky = prms['sky_coeffs']
traj_B = []
t0 = time.time()
ext_prev = float(ext_A)
for outer in range(N_SKY_OUTER):
    if SKY_SOLVER == 'cg':
        sky, loss = cal.fit_sky_cg(sky, gains, maxiter=CG_MAXITER,
                                   prior_sky=sky_init_full, prior_lambda=PRIOR_LAMBDA)
    else:
        sky, loss = cal.fit_sky_dirty(sky, gains, n_iter=N_SKY_INNER,
                                      step_size=SKY_STEP, beam_reg=3e-3)
    sky = jnp.array(project_positive(np.asarray(sky), sky_basis.A))
    gp_new, _ = cal.fit_gains_linear(sky)
    gains_new = {k: jnp.array(v) for k, v in
                 interp_gains_over_dead_channels({k: gp_new[k] for k in GAIN_KEYS}, dead).items()}
    ext_keep = cal.calc_reduced_chi2_excluding_chans({'sky_coeffs': sky, **gains}, exclude_chans=dead_chans, use_visibility_weights=True)
    ext_new  = cal.calc_reduced_chi2_excluding_chans({'sky_coeffs': sky, **gains_new}, exclude_chans=dead_chans, use_visibility_weights=True)
    if ext_new <= ext_keep:
        gains = gains_new
    # explicit gauge projection: re-pin the smooth per-channel gain amplitude
    # (the truly unconstrained mode) to the zero gauge, exchanging it into the
    # sky where the GSM prior adjudicates it — stops the sky/gain scale ratchet
    rp = cal.remove_degen({'sky_coeffs': sky, **gains}, which=('log_amp_sky',))
    sky, gains = rp['sky_coeffs'], {**gains, 'log_amp': rp['log_amp']}
    ext = float(min(ext_new, ext_keep))
    traj_B.append((time.time() - t0, ext))
    print(f'outer {outer}: loss={loss:.4e}  ext chi2={ext:.2f}'
          + ('' if ext_new <= ext_keep else '  [gain update rejected]'))
traj_B = np.array(traj_B)

## 6. Stage C — joint sky+beam L-BFGS polish

The original three-block alternating fit (`fit_joint_sky_beam_lbfgs`): closed-form gains,
then quasi-Newton over sky **and beam** with the diagonal-GN preconditioner. RFI weight
updates are disabled (`'rfi': 0`) because the soft-RFI machinery would slowly *resurrect*
the dead inpainted channels (their inpainted residuals look clean). The result is accepted
only if the external χ² improves — the v1 run showed this stage can trade external quality
for internal loss through the weakly-constrained beam.

In [ ]:
ext_B = float(traj_B[-1, 1])
final = {'sky_coeffs': sky, **gains}
prms_j = {'sky_coeffs': sky, 'beam_coeffs': jnp.array(beam_model.coeffs), **gains}
t0 = time.time()
prms_j, loss_l = cal.fit_joint_sky_beam_lbfgs(
    prms_j, n_outer=2, lbfgs_maxiter=LBFGS_MAXITER, lbfgs_history_size=30,
    solve_every={'gains': 1, 'rfi': 0}, verbose=True,
    verbose_exclude_chans=list(dead_chans))
dt_C = time.time() - t0
prms_j['sky_coeffs'] = jnp.array(project_positive(np.asarray(prms_j['sky_coeffs']), sky_basis.A))
ext_C = float(cal.calc_reduced_chi2_excluding_chans(prms_j, exclude_chans=dead_chans, use_visibility_weights=True))
if ext_C <= ext_B:
    final = prms_j
    sky = prms_j['sky_coeffs']
    gains = {k: prms_j[k] for k in GAIN_KEYS}
    print(f'joint lbfgs ACCEPTED: {dt_C:.0f}s, ext chi2 {ext_B:.2f} -> {ext_C:.2f}')
else:
    print(f'joint lbfgs REJECTED (ext chi2 {ext_B:.2f} -> {ext_C:.2f}); keeping stage-B result')

# CRITICAL for diagnostics: fit_joint_sky_beam_lbfgs re-solves gains internally
# WITHOUT dead-channel interpolation, leaving log_amp = 0 there. The model then
# shows the raw forward amplitude (~1000x the gain-calibrated scale) at those
# channels — two frequency deltas that bury the delay spectrum under a flat
# floor with a 1/(channel-spacing) comb. Re-interpolate before any diagnostics.
gi = interp_gains_over_dead_channels({k: np.asarray(final[k]) for k in GAIN_KEYS}, dead)
final = {**final, **{k: jnp.array(v) for k, v in gi.items()}}
rp = cal.remove_degen(final, which=('log_amp_sky',))
final = {**final, 'sky_coeffs': rp['sky_coeffs'], 'log_amp': rp['log_amp']}
sky = final['sky_coeffs']
gains = {k: final[k] for k in GAIN_KEYS}

# sky-departure diagnostic: the fit should refine the GSM, not replace it
mid_ch = NCHAN // 2
s_fit = np.asarray(final['sky_coeffs']) @ np.asarray(sky_basis.A)[mid_ch]
s_gsm = flux0[:, mid_ch]
cc = np.corrcoef(s_fit[keep], s_gsm[keep])[0, 1]
ratio = np.median(np.abs(s_fit[keep])) / np.median(s_gsm[keep])
print(f'fitted-vs-GSM on kept pixels: corr={cc:.3f}, median ratio={ratio:.2f} '
      + ('(sane)' if cc > 0.5 and 0.3 < ratio < 3 else '(DEPARTING — tighten PRIOR_LAMBDA / SKY_WEIGHT_CUT)'))

plt.figure(figsize=(8, 4))
plt.semilogy(traj_B[:, 0], traj_B[:, 1], 'o-', label='stage B (sky ' + SKY_SOLVER + ')')
plt.scatter([traj_B[-1, 0] + dt_C], [min(ext_C, ext_B)], color='r', zorder=5, label='after joint L-BFGS')
plt.axhline(1.0, color='k', ls=':', lw=0.8)
plt.xlabel('wall-clock seconds'); plt.ylabel('external reduced χ² (auto σ)')
plt.legend(); plt.title('convergence per second'); plt.tight_layout()

## 7. Diagnostics

### 7a. Per-channel χ² and weights

In [ ]:
model = np.asarray(cal.simulate(final))
dataP = np.asarray(cal.data)
resid = (dataP - model) * (data != 0)            # keep inpainted channels, drop true holes
chi2_f = np.nanmean(np.where(vis_weights > 0, np.abs(resid)**2 / sigma_eff[:, :, None]**2, np.nan), axis=(0, 2))
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax[0].semilogy(freqs/1e6, chi2_f); ax[0].axhline(1, color='k', ls=':')
ax[0].set_ylabel('χ²ν per channel (auto σ)')
ax[1].plot(freqs/1e6, np.exp(log_ch_w), label='channel weight (dead-chan mask)')
ax[1].plot(freqs/1e6, 1 - flags.mean(axis=(0, 2)), alpha=0.5, label='1 − pipeline flag frac')
ax[1].set_xlabel('MHz'); ax[1].legend(); fig.tight_layout()

### 7b. Residual delay spectra

Projected data vs projected residual, Blackman–Harris window, baseline-length bins.
Inpainted channels are *kept* here (they are pipeline-smooth; zeroing them imprints a comb),
and gains are re-interpolated across dead channels after Stage C — without that, the
internally re-solved gains leave the raw model amplitude at dead channels, and those two
frequency deltas alone produce a flat residual floor with a 1/(spur-spacing) ≈ 205 ns comb
that buries everything (this was the v2 'unphysical structure'). The model curve makes
model-injected delay structure (e.g. the nside-32 pixelization wedge) directly visible. What to look for: residual power at/beyond the horizon delay (dashed) on
short baselines → coupling; fixed-τ peaks independent of |b| → cable reflections;
band-position-dependent peaks (rerun with shifted `CH0`) → high-RM sources. The wedge-level
floor set by nside-32 pixelization (~few % of peak) is a known limit at this resolution —
rerun with `NSIDE_SKY=64` to push it down.

In [ ]:
from scipy.signal.windows import blackmanharris
win = blackmanharris(NCHAN); win /= win.mean()
delays = np.fft.fftshift(np.fft.fftfreq(NCHAN, d=dnu)) * 1e9
hole = (data == 0)
def dspec(x):
    return np.abs(np.fft.fftshift(np.fft.fft(win[None, :, None] * np.where(hole, 0, x), axis=1), axes=1))
ds_d, ds_m, ds_r = dspec(dataP), dspec(model), dspec(dataP - model)
bins = [(14, 30), (30, 60), (60, 120), (120, 290)]
fig, axes = plt.subplots(1, len(bins), figsize=(15, 3.6), sharey=True)
for ax, (lo, hi) in zip(axes, bins):
    m = (bl_len >= lo) & (bl_len < hi)
    ax.semilogy(delays, ds_d[:, :, m].mean(axis=(0, 2)), label='P(data)')
    ax.semilogy(delays, ds_m[:, :, m].mean(axis=(0, 2)), lw=0.8, alpha=0.7, label='model')
    ax.semilogy(delays, ds_r[:, :, m].mean(axis=(0, 2)), label='residual')
    tau_h = hi / 3e8 * 1e9
    ax.axvline(tau_h, color='k', ls='--', lw=0.8); ax.axvline(-tau_h, color='k', ls='--', lw=0.8)
    ax.set_title(f'{lo}-{hi} m (n={m.sum()})'); ax.set_xlabel('delay [ns]'); ax.set_xlim(-300, 300)
axes[0].set_ylabel('|V(τ)| [Jy]'); axes[0].legend(); fig.tight_layout()

### 7c. Recovered sky (positivity-projected)

Unfitted pixels show the GSM init (their flux was frozen and subtracted); fitted pixels
should look like a *refinement* of the GSM, not speckle — watch the corr/ratio printed in
Stage C.

In [ ]:
mid = NCHAN // 2
sky_map = np.asarray(sky) @ np.asarray(sky_basis.A)[mid]
fig = plt.figure(figsize=(12, 6))
healpy.mollview(np.log10(np.maximum(sky_map, 1e-3)), title=f'fitted sky, {freqs[mid]/1e6:.0f} MHz', min=0, max=3.0, sub=121, fig=fig)
healpy.mollview(np.log10(np.maximum(flux0[:, mid], 1e-3)), title='GSM+GLEAM init', min=0, max=3.0, sub=122, fig=fig)
print('negative pixels at mid-band: %.4f of pixels; most negative = %.3g vs peak %.3g'
      % (float((sky_map < 0).mean()), float(sky_map.min()), float(sky_map.max())))

## 8. Speed experiments

Moved to **`newnucal/benchmarks/benchmark_real_data_blocks.py`** — it mirrors this setup
via `newnucal.realdata` and times forward sim, loss+grad, dirty steps, gain solve and
(optionally) an L-BFGS outer:

```bash
cd newnucal/benchmarks
python benchmark_real_data_blocks.py --ntimes 4 --nchan 128            # this notebook's QUICK config
python benchmark_real_data_blocks.py --ntimes 8 --nchan 256 --lbfgs    # DEEP config
python benchmark_real_data_blocks.py --method 3d                       # 2D vs 3D path
```

## 9. Roadmap

1. ~~FR0 projector~~ — done (`project_out_time_mean=True`, tested).
2. **`NSIDE_SKY=64` rerun** — the nside-32 pixelization wedge floor (~few % of peak) is
   now the leading model artifact in the residual delay spectra.
3. **Coarse-ν forward model + band-limited interpolation** — attacks the dominant
   per-channel NUFFT cost; per-baseline coarse grids.
   (~~CG sky solver~~ — done: `fit_sky_cg`, preconditioned with the per-(pixel, ν)
   apparent-beam diagonal; `SKY_SOLVER` config above.)
4. **Gain-solve robustness** — replace the log-space phase-gradient surrogate with one or
   two Gauss-Newton steps on the true quadratic loss (the surrogate raised the loss when
   residual phases wrapped).
5. **Fagnoni CST beam basis** + beam fitting with a prior toward it (needed before
   `FIT_BEAM=True` is trustworthy at few-integration scale).
6. **Non-negative sky parameterization** (e.g. fit in sqrt-flux) to replace the POCS prox.
7. **More integrations** — the cheapest way to constrain the beam and dilute the FR0
   projector's 1/ntime DOF cost.
8. **Both polarizations / shifted bands** — discriminate instrumental vs sky residual
   structure.